# Chat with Docs UI

This notebook does these steps:

1. Get the URL of the MCP server on Modal (from the **MCP Server on Modal** notebook).
2. Make a Gradio chat app. The app uses the Gemini model `gemini-flash-lite-latest`.
3. Add a toggle: **Use the knowledge base**.
   - **On:** The app connects to the MCP server. Gemini can call the `search_documents` tool. Gemini gives the answer from the text of the documents.
   - **Off:** Gemini gives the answer from its general knowledge. The app does not use the MCP server.
     A second toggle, **Use web search**, lets Gemini call the `web_search` tool in this mode. It is on by default.

The app is the MCP client. The app gets the tool list from the MCP server and gives it to Gemini as function declarations.
When Gemini asks for a function call, the app calls the MCP tool and sends the result back to Gemini.
The `web_search` tool works in the same way. The app makes its function declaration and calls DuckDuckGo (no API key).

**Before you start:**

- Run the **Document Ingestion** notebook and the **MCP Server on Modal** notebook. The MCP server must be deployed.
- Add your Gemini API key to the `.env` file: `GEMINI_API_KEY=...`

**Code style:** The code follows the same ASD-STE100 rules as the other notebooks.

## 1. Install the packages

Do this step one time only.

In [ ]:
%pip install -q "google-genai==2.8.0" "gradio==6.15.2" "mcp==2.3.0" modal python-dotenv "ddgs==9.14.4"

## 2. Load the settings

The cell loads the `.env` file. The Gemini client gets the `GEMINI_API_KEY` value from the environment.

In [ ]:
import asyncio
import os
import weakref
from pathlib import Path

import gradio as gr
import modal
from ddgs import DDGS
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types
from mcp import Client

load_dotenv(Path(".env"))

MODEL = "gemini-flash-lite-latest"
APP_NAME = "workshop-documents-mcp"   # The same name as in mcp_server.py.
MAX_TOOL_ROUNDS = 5                   # Largest number of tool rounds for one question.
MAX_SEARCH_RESULTS = 5                # Number of web results for one search.

assert os.environ.get("GEMINI_API_KEY"), "Add GEMINI_API_KEY to the .env file."
gemini = genai.Client()

## 3. Get the URL of the MCP server

The MCP endpoint is the web URL of the Modal function plus the path `/mcp`.
If the `.env` file has an `MCP_URL` value, the cell uses that value.

In [ ]:
def get_mcp_url():
    """Get the URL of the deployed MCP server."""
    if os.environ.get("MCP_URL"):
        return os.environ["MCP_URL"]
    return modal.Function.from_name(APP_NAME, "serve").get_web_url() + "/mcp"


MCP_URL = get_mcp_url()
print(f"MCP URL: {MCP_URL}")

## 4. Make the tools for Gemini

The functions below do these steps:

- `to_gemini_tool` changes the MCP tool list into one Gemini tool with function declarations.
  The MCP input schema is a JSON schema. Thus, the declaration uses `parameters_json_schema`.
- `call_mcp_tool` calls the MCP tool and gives the text of the result.
- `WEB_SEARCH_TOOL` is the function declaration of the `web_search` tool.
- `call_web_search` searches DuckDuckGo and gives the title, URL and text of each result.

In [ ]:
def to_gemini_tool(mcp_tools):
    """Change the MCP tools into one Gemini tool."""
    declarations = [
        types.FunctionDeclaration(
            name=tool.name,
            description=tool.description or "",
            parameters_json_schema=tool.input_schema,
        )
        for tool in mcp_tools
    ]
    return types.Tool(function_declarations=declarations)


async def call_mcp_tool(mcp, function_call):
    """Call the MCP tool that Gemini asks for. Return the list of text chunks and the error flag."""
    result = await mcp.call_tool(function_call.name, dict(function_call.args or {}))
    texts = [content.text for content in result.content if content.type == "text"]
    return texts, result.is_error


WEB_SEARCH_TOOL = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="web_search",
        description="Search the web. Use this tool for current or specific facts.",
        parameters_json_schema={
            "type": "object",
            "properties": {"query": {"type": "string", "description": "The search words."}},
            "required": ["query"],
        },
    ),
])


async def call_web_search(function_call):
    """Search the web for the query that Gemini gives. Return the list of results and the error flag."""
    query = (function_call.args or {}).get("query", "").strip()
    if not query:
        return ["The query is empty. Give the search words in the query argument."], True
    try:
        # DDGS is not async. Thus, run the search in a thread.
        results = await asyncio.to_thread(DDGS().text, query, max_results=MAX_SEARCH_RESULTS)
    except Exception as error:
        return [f"The search failed: {error}"], True
    texts = [f"{result['title']}\n{result['href']}\n{result['body']}" for result in results]
    return texts, False


async with Client(MCP_URL) as mcp:
    tools = await mcp.list_tools()
print(to_gemini_tool(tools.tools).model_dump_json(indent=2, exclude_none=True))
print(WEB_SEARCH_TOOL.model_dump_json(indent=2, exclude_none=True))

## 5. Write the chat function

The function `stream_answer` does these steps:

1. Make the Gemini contents from the chat history.
2. If the knowledge base toggle is on, connect to the MCP server and give its tools to Gemini.
3. If the knowledge base toggle is off and the web search toggle is on, give the `web_search` tool to Gemini.
4. If the two toggles are off, stream the answer from Gemini without tools.
5. Stream the answer. If Gemini asks for function calls, call the tools, send the results to Gemini, and stream again.
6. Show each tool call to the user. A tool call shows the query and the text that the tool gave.

The function `run_tool_loop` does step 5 for the two tools. The MCP tools and the web search use the same loop.
The two system instructions tell Gemini how to answer in each mode.

In [ ]:
SYSTEM_WITH_DOCUMENTS = (
    "You answer questions about the workshop policy documents. "
    "Use the search_documents tool to find the text that answers the question. "
    "Give the answer only from the text that the tool gives. "
    "If the text does not contain the answer, say so. "
    "You can search more than one time with different words."
)

SYSTEM_WITHOUT_DOCUMENTS = (
    "You are a helpful assistant. Give the answer from your general knowledge. "
    "You do not have access to the workshop documents."
)


def make_contents(history, message):
    """Make the Gemini contents from the Gradio history and the new message."""
    contents = []
    for item in history:
        # Do not send the tool-call notes. They are only for the user.
        if (item.get("metadata") or {}).get("title"):
            continue
        content = item["content"]
        # Gradio 6 keeps the content as a list of blocks. Keep only the text blocks.
        if isinstance(content, list):
            content = "".join(block.get("text", "") for block in content if block.get("type") == "text")
        if item["role"] in ("user", "assistant") and isinstance(content, str) and content:
            role = "model" if item["role"] == "assistant" else "user"
            contents.append(types.Content(role=role, parts=[types.Part(text=content)]))
    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))
    return contents


SEARCH_INSTRUCTION = " Use the web_search tool when the question needs current or specific facts."


def make_config(use_documents, gemini_tool=None, use_web_search=True):
    """Make the Gemini configuration for each mode."""
    if not use_documents and not use_web_search:
        return types.GenerateContentConfig(system_instruction=SYSTEM_WITHOUT_DOCUMENTS)
    if not use_documents:
        return types.GenerateContentConfig(
            system_instruction=SYSTEM_WITHOUT_DOCUMENTS + SEARCH_INSTRUCTION,
            tools=[WEB_SEARCH_TOOL],
            # The app calls the web search. Thus, turn off the automatic function calls of the SDK.
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        )
    return types.GenerateContentConfig(
        system_instruction=SYSTEM_WITH_DOCUMENTS,
        tools=[gemini_tool],
        # The app calls the MCP tools. Thus, turn off the automatic function calls of the SDK.
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )


def tool_note(function_call):
    """Make a chat message that shows a tool call."""
    query = (function_call.args or {}).get("query", "")
    if function_call.name == "web_search":
        title = f"🌐 Searched the web: {query}"
    else:
        title = f"🔎 Searched the documents: {query}"
    return gr.ChatMessage(role="assistant", content="", metadata={"title": title, "status": "pending"})


GEMINI_CLIENTS = weakref.WeakKeyDictionary()


def gemini_aio():
    """Get the async Gemini client of the running event loop.

    The test cell and the Gradio app use different event loops. An async client works in one loop only.
    """
    loop = asyncio.get_running_loop()
    if loop not in GEMINI_CLIENTS:
        GEMINI_CLIENTS[loop] = genai.Client().aio
    return GEMINI_CLIENTS[loop]


async def stream_turn(contents, config, answer, notes):
    """Stream one Gemini response. Return all the parts of the response."""
    parts = []
    stream = await gemini_aio().models.generate_content_stream(model=MODEL, contents=contents, config=config)
    async for chunk in stream:
        if not chunk.candidates or not chunk.candidates[0].content or not chunk.candidates[0].content.parts:
            continue
        for part in chunk.candidates[0].content.parts:
            parts.append(part)
            if part.text and not part.thought:
                answer.content += part.text
                yield parts
    yield parts


async def run_tool_loop(contents, config, answer, notes, call_tool):
    """Stream the answer. Call the tools with call_tool when Gemini asks for them."""
    for _ in range(MAX_TOOL_ROUNDS):
        async for parts in stream_turn(contents, config, answer, notes):
            yield
        calls = [part.function_call for part in parts if part.function_call]
        if not calls:
            return
        # Keep all the parts. The parts can have thought signatures that Gemini must get back.
        contents.append(types.Content(role="model", parts=parts))
        responses = []
        for call in calls:
            note = tool_note(call)
            notes.append(note)
            yield
            texts, is_error = await call_tool(call)
            note.content = "\n\n---\n\n".join(texts) or "The tool gave no text."
            note.metadata["status"] = "done"
            yield
            key = "error" if is_error else "result"
            responses.append(types.Part(function_response=types.FunctionResponse(
                id=call.id, name=call.name, response={key: texts},
            )))
        contents.append(types.Content(role="user", parts=responses))
    answer.content += "\n\n*The app stopped after the largest number of tool rounds.*"


async def answer_with_documents(contents, answer, notes):
    """Stream the answer with the MCP tools of the knowledge base."""
    # Gradio can run each step of this generator in a different task.
    # The MCP client must open and close in the same task. Thus, run the MCP client in one task
    # and send the updates to this generator through a queue.
    updates = asyncio.Queue()
    done = object()

    async def run():
        try:
            async with Client(MCP_URL) as mcp:
                tools = await mcp.list_tools()
                config = make_config(True, to_gemini_tool(tools.tools))
                async for _ in run_tool_loop(contents, config, answer, notes, lambda call: call_mcp_tool(mcp, call)):
                    await updates.put(None)
        finally:
            await updates.put(done)

    task = asyncio.create_task(run())
    try:
        while await updates.get() is not done:
            yield
        await task   # Raise the error of the task, if there is an error.
    finally:
        task.cancel()


def first_error(error):
    """Get the first real error from an error group (the MCP client can give error groups)."""
    while getattr(error, "exceptions", None):
        error = error.exceptions[0]
    return error


async def stream_answer(message, history, use_documents, use_web_search=True):
    """Stream the answer from Gemini. Show the tool calls to the user."""
    contents = make_contents(history, message)
    notes = []
    answer = gr.ChatMessage(role="assistant", content="")
    try:
        if use_documents:
            turns = answer_with_documents(contents, answer, notes)
        elif use_web_search:
            turns = run_tool_loop(contents, make_config(False), answer, notes, call_web_search)
        else:
            turns = stream_turn(contents, make_config(False, use_web_search=False), answer, notes)
        async for _ in turns:
            yield notes + [answer]
    except errors.APIError as error:
        answer.content += f"\n\n**Gemini error {error.code}:** {error.message}"
    except Exception as error:
        error = first_error(error)
        answer.content += f"\n\n**Error:** {type(error).__name__}: {error}"
    if not answer.content:
        answer.content = "*Gemini gave no text.*"
    yield notes + [answer]

## 6. Do a test of the chat function

The test asks one question in each mode. The test shows the tool calls and the start of each answer.

In [ ]:
TEST_QUESTION = "What are the principles of the IT architecture framework?"

for use_documents in (True, False):
    last = None
    async for last in stream_answer(TEST_QUESTION, [], use_documents):
        pass
    print(f"USE THE KNOWLEDGE BASE: {use_documents}")
    for item in last:
        if item.metadata.get("title"):
            print(f"  {item.metadata['title']}")
    print(f"  ANSWER: {last[-1].content[:500]}...")
    print()

## 7. Start the Gradio app

The two toggles are below the chat box, in **Additional Inputs**.

The app opens in the notebook. To open the app in a browser, go to the local URL that the cell shows.
To share the app with other persons, set `share=True`.

In [ ]:
EXAMPLES = [
    ["What are the objectives of the financial technology development policy?", True, True],
    ["What are the goals of the e-services for business strategy?", True, True],
    ["What is an IT architecture framework in general?", False, False],
    ["What is the latest news about digital government in Cambodia?", False, True],
]

demo = gr.ChatInterface(
    fn=stream_answer,
    title="Chat with the Workshop Documents",
    description=(
        "Turn on **Use the knowledge base** to get the answer from the documents (MCP server on Modal). "
        "Turn it off to get the answer from the general knowledge of Gemini. "
        "**Use web search** lets Gemini search the web when the knowledge base is off."
    ),
    additional_inputs=[
        gr.Checkbox(value=True, label="Use the knowledge base (MCP server)"),
        gr.Checkbox(value=True, label="Use web search (when the knowledge base is off)"),
    ],
    additional_inputs_accordion=gr.Accordion("Additional Inputs", open=True),
    examples=EXAMPLES,
    cache_examples=False,
)

demo.launch()

## 8. Stop the app

Run this cell to stop the Gradio server.

In [ ]:
demo.close()